In [7]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from notebookutils import *
from pyspark.sql import functions as F
from delta.tables import *
from notebookutils import mssparkutils

StatementMeta(, 301bddea-4d83-4745-9d9c-2d8743e84037, 9, Finished, Available, Finished, False)

In [8]:
tables_str = tables_str
lakehouse = lakehouse
workspace = workspace
silver = silver
gold = gold

# tables_str = "Patient,Encounter,Observation,Condition"
# gold = "gold"
# silver = "silver"
# workspace = "Fabric_Assignment_Synapx"
# lakehouse = "FHIR_Lakehouse"

Tables = [t.strip() for t in tables_str.split(",")]

StatementMeta(, 301bddea-4d83-4745-9d9c-2d8743e84037, 10, Finished, Available, Finished, False)

In [9]:
table = None
for i in Tables:
    silver_path = f"Files/{silver}/{i.lower()}"
    if i.lower() == "patient": 
        patient_df = spark.read.format("delta").load(silver_path).filter("is_current = true")
    elif i.lower() == "encounter":
        encounter_df = spark.read.format("delta").load(silver_path).filter("is_current = true")
    elif i.lower() == "observation":
        observation_df = spark.read.format("delta").load(silver_path).filter("is_current = true")
    elif i.lower() == "condition":
        condition_df = spark.read.format("delta").load(silver_path).filter("is_current = true")


StatementMeta(, 301bddea-4d83-4745-9d9c-2d8743e84037, 11, Finished, Available, Finished, False)

In [10]:
gold_dim_patient = (
    patient_df
    .select(
        "patient_id",
        "given_name",
        "family_name",
        "gender"
    )
)

gold_dim_patient.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Files/{gold}/patient_dim")


StatementMeta(, 301bddea-4d83-4745-9d9c-2d8743e84037, 12, Finished, Available, Finished, False)

In [11]:
gold_dim_encounter = (
    encounter_df
    .select(
        "class_code",
        "class_display",
        "type_code",
        "type_display"
    )
    .dropDuplicates()
)

gold_dim_encounter.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Files/{gold}/dim_encounter_type")

StatementMeta(, 301bddea-4d83-4745-9d9c-2d8743e84037, 13, Finished, Available, Finished, False)

In [12]:
gold_fact_encounter = (
    encounter_df
    .select(
        "encounter_id",
        "patient_id",
        "class_code",
        "type_code",
        "status",
        "start_date",
        "end_date"
    )
)

gold_fact_encounter.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Files/{gold}/fact_encounter")

StatementMeta(, 301bddea-4d83-4745-9d9c-2d8743e84037, 14, Finished, Available, Finished, False)

In [13]:
gold_fact_observation = (
    observation_df
    .select(
        "observation_id",
        "patient_id",
        "test_name",
        "value",
        "unit",
        "status"
    )
)

gold_fact_observation.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Files/{gold}/fact_observation")

StatementMeta(, 301bddea-4d83-4745-9d9c-2d8743e84037, 15, Finished, Available, Finished, False)

In [14]:
gold_fact_condition = (
    condition_df
    .select(
        "resource_id",
        "patient_id",
        "encounter_id",
        "condition_text"
    )
)
    
gold_fact_condition.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Files/{gold}/fact_condition")

StatementMeta(, 301bddea-4d83-4745-9d9c-2d8743e84037, 16, Finished, Available, Finished, False)

# Relationships (Power BI / Warehouse)

dim_patient.patient_id → fact_encounter.patient_id,

dim_patient.patient_id → fact_observation.patient_id,

dim_patient.patient_id → fact_condition.patient_id,

dim_encounter_type.class_code → fact_encounter.class_code
